In [0]:
%sql
CREATE TABLE dev.mohit_gangwani.tivo_national_dual_stations AS
WITH ism AS (
  SELECT inscape_station_id
  , inscape_call_sign
  , mapped_vendor
  , mapped_vendor_station_id
  FROM (
    SELECT inscape_station_id
    , inscape_call_sign
    , mapped_vendor
    , mapped_vendor_station_id
    , ROW_NUMBER() OVER (PARTITION BY mapped_vendor, mapped_vendor_station_id ORDER BY ism.created_at DESC) AS rn
    FROM prod.detection.inscape_station_map ism
    JOIN prod.detection.epg_station st
      ON st.station_id = ism.mapped_vendor_station_id
     AND st.vendor_name = ism.mapped_vendor
     AND (st.ingested = 'TRUE' OR st.attributed = 'TRUE')
  ) ism
  WHERE ism.rn = 1
)
, dual_station AS (
  SELECT REPLACE(w.inscape_station_name, ' Network', '') AS station_name, COUNT(DISTINCT w.station_time_zone) AS ttl_tz, COUNT(*) AS station_count
  FROM detection.epg_station w
  JOIN ism
    ON ism.mapped_vendor_station_id = w.station_id
   AND ism.mapped_vendor = w.vendor_name
  WHERE (w.ingested = 'TRUE' OR w.attributed = 'TRUE')
    AND w.vendor_name = 'TIVO'
    AND w.local_or_national = 'National'
GROUP BY 1
)
SELECT st.inscape_station_name
, SPLIT_PART(st.station_time_zone, ' ', 1) AS station_tz
, st.station_id
, ism.inscape_call_sign AS station_call_sign
FROM prod.detection.epg_station st
JOIN ism
  ON ism.mapped_vendor_station_id = st.station_id
 AND ism.mapped_vendor = st.vendor_name
JOIN dual_station ds
  ON ds.station_name = REPLACE(st.inscape_station_name, ' Network', '')
 AND ds.station_count > 1
 AND ds.ttl_tz > 1
WHERE (st.ingested = 'TRUE' OR st.attributed = 'TRUE')
  AND st.vendor_name = 'TIVO'
  AND st.local_or_national = 'National'
  AND st.cleaned_station_name != 'IND'
  AND st.cleaned_station_name NOT LIKE 'FanDuel%'
GROUP BY ALL
ORDER BY 1, 2

In [0]:
%sql
UPDATE dev.mohit_gangwani.tivo_national_dual_stations
SET inscape_station_name = 'Women\'s Entertainment Network'
WHERE inscape_station_name = 'Women\'s Entertainment'

In [0]:
%sql
WITH ism AS (
  SELECT inscape_station_id
  , inscape_call_sign
  , mapped_vendor
  , mapped_vendor_station_id
  FROM (
    SELECT inscape_station_id
    , inscape_call_sign
    , mapped_vendor
    , mapped_vendor_station_id
    , ROW_NUMBER() OVER (PARTITION BY mapped_vendor, mapped_vendor_station_id ORDER BY ism.created_at DESC) AS rn
    FROM prod.detection.inscape_station_map ism
    JOIN prod.detection.epg_station st
      ON st.station_id = ism.mapped_vendor_station_id
     AND st.vendor_name = ism.mapped_vendor
     AND (st.ingested = 'TRUE' OR st.attributed = 'TRUE')
  ) ism
  WHERE ism.rn = 1
)
, dual_station AS (
  SELECT REPLACE(w.inscape_station_name, ' Network', '') AS station_name, COUNT(DISTINCT w.station_time_zone) AS ttl_tz, COUNT(*) AS station_count
  FROM detection.epg_station w
  JOIN ism
    ON ism.mapped_vendor_station_id = w.station_id
   AND ism.mapped_vendor = w.vendor_name
  WHERE (w.ingested = 'TRUE' OR w.attributed = 'TRUE')
    AND w.vendor_name = 'TIVO'
    AND w.local_or_national = 'National'
GROUP BY 1
)
SELECT st.inscape_station_name
, SPLIT_PART(st.station_time_zone, ' ', 1) AS station_tz
, st.station_id
, ism.inscape_call_sign AS station_call_sign
, CASE WHEN ds.station_count > 1 AND ds.ttl_tz > 1 THEN 1 END AS dual_station
FROM prod.detection.epg_station st
JOIN ism
  ON ism.mapped_vendor_station_id = st.station_id
 AND ism.mapped_vendor = st.vendor_name
JOIN dual_station ds
  ON ds.station_name = REPLACE(st.inscape_station_name, ' Network', '')
WHERE (st.ingested = 'TRUE' OR st.attributed = 'TRUE')
  AND st.vendor_name = 'TIVO'
  AND st.local_or_national = 'National'
  AND st.cleaned_station_name != 'IND'
  AND st.cleaned_station_name NOT LIKE 'FanDuel%'
GROUP BY ALL
ORDER BY 1, 2

In [0]:
%sql
WITH simulcast_schedule AS (
  WITH dual_station_schedule AS (
    SELECT sch.airdate
    , ism.inscape_station_name
    , ism.station_call_sign
    , ism.station_tz
    , sh.database_key AS epid
    FROM prod.detection.epg_schedule sch
    JOIN dev.mohit_gangwani.tivo_national_dual_stations ism
      ON ism.station_id = sch.fk_station_id
    JOIN prod.detection.epg_show sh
      ON sh.show_id = sch.fk_show_id
     AND sh.vendor_name = 'TIVO'
    WHERE sch.airdate >= CURRENT_DATE - 90
      AND sch.vendor_name = 'TIVO'
    GROUP BY ALL
  )
  SELECT a.airdate
  , a.inscape_station_name
  , a.station_call_sign
  , a.station_tz
  , a.epid
  FROM dual_station_schedule a
  JOIN dual_station_schedule b
    ON a.epid = b.epid
   AND a.airdate = b.airdate
   AND a.inscape_station_name != b.inscape_station_name
   AND a.station_call_sign != b.station_call_sign
  GROUP BY ALL
)


In [0]:
%sql
(
  SELECT ism.inscape_station_name
  , 'DP4' AS pipeline
  , COUNT(*)*1.0 AS sessions_count
  , SUM(TIMESTAMPDIFF(SECOND, ts_start, ts_end))/3600.0 AS total_duration
  , COUNT(Distinct vc.tvid) AS total_tvs
  FROM (
    SELECT *
    , LEAD(ts_start) OVER (PARTITION BY tvid ORDER BY ts_start, ts_end) AS next_start
    , LAG(ts_end) OVER (PARTITION BY tvid ORDER BY ts_start, ts_end) AS prev_end
    , LEAD(ts_duration) OVER (PARTITION BY tvid ORDER BY ts_start, ts_end) AS next_duration
    , LAG(ts_duration) OVER (PARTITION BY tvid ORDER BY ts_start, ts_end) AS prev_duration
    , CASE WHEN next_start < ts_end AND DATE_TRUNC('HOUR', next_start) = DATE_TRUNC('HOUR', ts_end) THEN 1
           WHEN prev_end > ts_start AND DATE_TRUNC('HOUR', prev_end) = DATE_TRUNC('HOUR', ts_start) THEN 1
      END AS overlapping_session
    , CASE WHEN overlapping_session IS NULL THEN 1
           ELSE CASE WHEN next_start IS NOT NULL AND next_start < ts_end AND next_duration < ts_duration AND DATE_TRUNC('HOUR', next_start) = DATE_TRUNC('HOUR', ts_start) THEN 1
                     WHEN prev_end IS NOT NULL AND prev_end > ts_start AND prev_duration < ts_duration AND DATE_TRUNC('HOUR', prev_end) = DATE_TRUNC('HOUR', ts_end) THEN 1
        END END AS keep
    FROM (
      SELECT tvid
      , ts_start
      , ts_end
      , TIMESTAMPDIFF(SECOND, ts_start, ts_end) AS ts_duration
      , cid
      , chan_callsign
      , epid
      , ROW_NUMBER() OVER (PARTITION BY tvid, ts_start ORDER BY ts_start, ts_end DESC, epid) AS rn
      FROM prod.staging.vizio_content_firehose
      WHERE ts_start >= '2025-05-11 11:00:00'
            AND ts_start < CURRENT_TIMESTAMP - INTERVAL 5 HOURS
      
    )
    WHERE rn = 1
  ) vc
  JOIN dev.mohit_gangwani.tivo_national_dual_stations ism
    ON ism.station_call_sign = vc.chan_callsign
  WHERE keep = 1
    AND DATE_TRUNC('HOUR', ts_start) != '2025-05-13 21:00:00'
  GROUP BY 1
)
UNION
(
  SELECT ism.inscape_station_name
  , 'DP5' AS pipeline
  , COUNT(*)*1.0 AS sessions_count
  , SUM(TIMESTAMPDIFF(SECOND, ts_start, ts_end))/3600.0 AS total_duration
  , COUNT(Distinct vc.tvid) AS total_tvs
  FROM (
    SELECT *
    , LEAD(ts_start) OVER (PARTITION BY tvid ORDER BY ts_start, ts_end) AS next_start
    , LAG(ts_end) OVER (PARTITION BY tvid ORDER BY ts_start, ts_end) AS prev_end
    , LEAD(ts_duration) OVER (PARTITION BY tvid ORDER BY ts_start, ts_end) AS next_duration
    , LAG(ts_duration) OVER (PARTITION BY tvid ORDER BY ts_start, ts_end) AS prev_duration
    , CASE WHEN next_start < ts_end AND DATE_TRUNC('HOUR', next_start) = DATE_TRUNC('HOUR', ts_end) THEN 1
           WHEN prev_end > ts_start AND DATE_TRUNC('HOUR', prev_end) = DATE_TRUNC('HOUR', ts_start) THEN 1
      END AS overlapping_session
    , CASE WHEN overlapping_session IS NULL THEN 1
           ELSE CASE WHEN next_start IS NOT NULL AND next_start < ts_end AND next_duration < ts_duration AND DATE_TRUNC('HOUR', next_start) = DATE_TRUNC('HOUR', ts_start) THEN 1
                     WHEN prev_end IS NOT NULL AND prev_end > ts_start AND prev_duration < ts_duration AND DATE_TRUNC('HOUR', prev_end) = DATE_TRUNC('HOUR', ts_end) THEN 1
        END END AS keep
    FROM (
      SELECT tvid
      , ts_start
      , ts_end
      , TIMESTAMPDIFF(SECOND, ts_start, ts_end) AS ts_duration
      , cid
      , chan_callsign
      , epid
      , ROW_NUMBER() OVER (PARTITION BY tvid, ts_start ORDER BY ts_start, ts_end DESC, epid) AS rn
      FROM prod.cooker.vizio_content_firehose vc
      WHERE ts_start >= '2025-05-11 11:00:00'
            AND ts_start < CURRENT_TIMESTAMP - INTERVAL 5 HOURS
      
    )
    WHERE rn = 1
  ) vc
  JOIN dev.mohit_gangwani.tivo_national_dual_stations ism
    ON ism.station_call_sign = vc.chan_callsign
  WHERE keep = 1
    AND DATE_TRUNC('HOUR', ts_start) != '2025-05-13 21:00:00'
  GROUP BY 1
)

In [0]:
%sql
WITH simulcast_schedule AS (
  WITH dual_station_schedule AS (
    SELECT sch.airdate
    , ism.inscape_station_name
    , ism.station_call_sign
    , ism.station_tz
    , sh.database_key AS epid
    FROM prod.detection.epg_schedule sch
    JOIN dev.mohit_gangwani.tivo_national_dual_stations ism
      ON ism.station_id = sch.fk_station_id
    JOIN prod.detection.epg_show sh
      ON sh.show_id = sch.fk_show_id
     AND sh.vendor_name = 'TIVO'
    WHERE sch.airdate >= CURRENT_DATE - 90
      AND sch.vendor_name = 'TIVO'
    GROUP BY ALL
  )
  SELECT a.airdate
  , a.inscape_station_name
  , a.station_call_sign
  , a.station_tz
  , a.epid
  FROM dual_station_schedule a
  JOIN dual_station_schedule b
    ON a.epid = b.epid
   AND a.airdate = b.airdate
   AND a.inscape_station_name = b.inscape_station_name
   AND a.station_call_sign != b.station_call_sign
  GROUP BY ALL
)
(
  SELECT ssch.inscape_station_name
  , ssch.station_tz
  , 'DP4' AS pipeline
  , COUNT(*)*1.0 AS sessions_count
  , SUM(TIMESTAMPDIFF(SECOND, ts_start, ts_end))/3600.0 AS total_duration
  , COUNT(Distinct vc.tvid) AS total_tvs
  FROM (
    SELECT *
    , LEAD(ts_start) OVER (PARTITION BY tvid ORDER BY ts_start, ts_end) AS next_start
    , LAG(ts_end) OVER (PARTITION BY tvid ORDER BY ts_start, ts_end) AS prev_end
    , LEAD(ts_duration) OVER (PARTITION BY tvid ORDER BY ts_start, ts_end) AS next_duration
    , LAG(ts_duration) OVER (PARTITION BY tvid ORDER BY ts_start, ts_end) AS prev_duration
    , CASE WHEN next_start < ts_end AND DATE_TRUNC('HOUR', next_start) = DATE_TRUNC('HOUR', ts_end) THEN 1
           WHEN prev_end > ts_start AND DATE_TRUNC('HOUR', prev_end) = DATE_TRUNC('HOUR', ts_start) THEN 1
      END AS overlapping_session
    , CASE WHEN overlapping_session IS NULL THEN 1
           ELSE CASE WHEN next_start IS NOT NULL AND next_start < ts_end AND next_duration < ts_duration AND DATE_TRUNC('HOUR', next_start) = DATE_TRUNC('HOUR', ts_start) THEN 1
                     WHEN prev_end IS NOT NULL AND prev_end > ts_start AND prev_duration < ts_duration AND DATE_TRUNC('HOUR', prev_end) = DATE_TRUNC('HOUR', ts_end) THEN 1
        END END AS keep
    FROM (
      SELECT tvid
      , ts_start
      , ts_end
      , TIMESTAMPDIFF(SECOND, ts_start, ts_end) AS ts_duration
      , cid
      , chan_callsign
      , epid
      , air_date
      , ROW_NUMBER() OVER (PARTITION BY tvid, ts_start ORDER BY ts_start, ts_end DESC, epid) AS rn
      FROM prod.staging.vizio_content_firehose
      WHERE ts_start >= '2025-05-11 11:00:00'
        AND ts_start < CURRENT_TIMESTAMP - INTERVAL 5 HOURS
    )
    WHERE rn = 1
  ) vc
  JOIN simulcast_schedule ssch
    ON ssch.station_call_sign = vc.chan_callsign
   AND ssch.airdate = vc.air_date
   AND ssch.epid = vc.epid
  WHERE keep = 1
    AND DATE_TRUNC('HOUR', ts_start) != '2025-05-13 21:00:00'
  GROUP BY 1, 2
)
UNION
(
  SELECT ssch.inscape_station_name
  , ssch.station_tz
  , 'DP5' AS pipeline
  , COUNT(*)*1.0 AS sessions_count
  , SUM(TIMESTAMPDIFF(SECOND, ts_start, ts_end))/3600.0 AS total_duration
  , COUNT(Distinct vc.tvid) AS total_tvs
  FROM (
    SELECT *
    , LEAD(ts_start) OVER (PARTITION BY tvid ORDER BY ts_start, ts_end) AS next_start
    , LAG(ts_end) OVER (PARTITION BY tvid ORDER BY ts_start, ts_end) AS prev_end
    , LEAD(ts_duration) OVER (PARTITION BY tvid ORDER BY ts_start, ts_end) AS next_duration
    , LAG(ts_duration) OVER (PARTITION BY tvid ORDER BY ts_start, ts_end) AS prev_duration
    , CASE WHEN next_start < ts_end AND DATE_TRUNC('HOUR', next_start) = DATE_TRUNC('HOUR', ts_end) THEN 1
           WHEN prev_end > ts_start AND DATE_TRUNC('HOUR', prev_end) = DATE_TRUNC('HOUR', ts_start) THEN 1
      END AS overlapping_session
    , CASE WHEN overlapping_session IS NULL THEN 1
           ELSE CASE WHEN next_start IS NOT NULL AND next_start < ts_end AND next_duration < ts_duration AND DATE_TRUNC('HOUR', next_start) = DATE_TRUNC('HOUR', ts_start) THEN 1
                     WHEN prev_end IS NOT NULL AND prev_end > ts_start AND prev_duration < ts_duration AND DATE_TRUNC('HOUR', prev_end) = DATE_TRUNC('HOUR', ts_end) THEN 1
        END END AS keep
    FROM (
      SELECT tvid
      , ts_start
      , ts_end
      , TIMESTAMPDIFF(SECOND, ts_start, ts_end) AS ts_duration
      , cid
      , chan_callsign
      , epid
      , air_date
      , ROW_NUMBER() OVER (PARTITION BY tvid, ts_start ORDER BY ts_start, ts_end DESC, epid) AS rn
      FROM prod.cooker.vizio_content_firehose vc
      WHERE ts_start >= '2025-05-11 11:00:00'
        AND ts_start < CURRENT_TIMESTAMP - INTERVAL 5 HOURS
    )
    WHERE rn = 1
  ) vc
  JOIN simulcast_schedule ssch
    ON ssch.station_call_sign = vc.chan_callsign
   AND ssch.airdate = vc.air_date
   AND ssch.epid = vc.epid
  WHERE keep = 1
    AND DATE_TRUNC('HOUR', ts_start) != '2025-05-13 21:00:00'
  GROUP BY 1, 2
)

Databricks visualization. Run in Databricks to view.

In [0]:
%sql
(
  SELECT 'DP4' AS pipeline
  , COUNT(*)*1.0 AS sessions_count
  , SUM(TIMESTAMPDIFF(SECOND, ts_start, ts_end))/3600.0 AS total_duration
  , COUNT(Distinct vc.tvid) AS total_tvs
  FROM (
    SELECT *
    , LEAD(ts_start) OVER (PARTITION BY tvid ORDER BY ts_start, ts_end) AS next_start
    , LAG(ts_end) OVER (PARTITION BY tvid ORDER BY ts_start, ts_end) AS prev_end
    , LEAD(ts_duration) OVER (PARTITION BY tvid ORDER BY ts_start, ts_end) AS next_duration
    , LAG(ts_duration) OVER (PARTITION BY tvid ORDER BY ts_start, ts_end) AS prev_duration
    , CASE WHEN next_start < ts_end AND DATE_TRUNC('HOUR', next_start) = DATE_TRUNC('HOUR', ts_end) THEN 1
           WHEN prev_end > ts_start AND DATE_TRUNC('HOUR', prev_end) = DATE_TRUNC('HOUR', ts_start) THEN 1
      END AS overlapping_session
    , CASE WHEN overlapping_session IS NULL THEN 1
           ELSE CASE WHEN next_start IS NOT NULL AND next_start < ts_end AND next_duration < ts_duration AND DATE_TRUNC('HOUR', next_start) = DATE_TRUNC('HOUR', ts_start) THEN 1
                     WHEN prev_end IS NOT NULL AND prev_end > ts_start AND prev_duration < ts_duration AND DATE_TRUNC('HOUR', prev_end) = DATE_TRUNC('HOUR', ts_end) THEN 1
        END END AS keep
    FROM (
      SELECT tvid
      , ts_start
      , ts_end
      , TIMESTAMPDIFF(SECOND, ts_start, ts_end) AS ts_duration
      , cid
      , chan_callsign
      , epid
      , ROW_NUMBER() OVER (PARTITION BY tvid, ts_start ORDER BY ts_start, ts_end DESC, epid) AS rn
      FROM prod.staging.vizio_content_firehose
      WHERE ts_start >= '2025-05-11 11:00:00'
            AND ts_start < CURRENT_TIMESTAMP - INTERVAL 5 HOURS
      
    )
    WHERE rn = 1
  ) vc
  JOIN prod.detection.epg_station ism
    ON ism.station_call_sign = vc.chan_callsign
   AND ism.local_or_national = 'National'
  WHERE keep = 1
    AND DATE_TRUNC('HOUR', ts_start) != '2025-05-13 21:00:00'
  GROUP BY 1
)
UNION
(
  SELECT 'DP5' AS pipeline
  , COUNT(*)*1.0 AS sessions_count
  , SUM(TIMESTAMPDIFF(SECOND, ts_start, ts_end))/3600.0 AS total_duration
  , COUNT(Distinct vc.tvid) AS total_tvs
  FROM (
    SELECT *
    , LEAD(ts_start) OVER (PARTITION BY tvid ORDER BY ts_start, ts_end) AS next_start
    , LAG(ts_end) OVER (PARTITION BY tvid ORDER BY ts_start, ts_end) AS prev_end
    , LEAD(ts_duration) OVER (PARTITION BY tvid ORDER BY ts_start, ts_end) AS next_duration
    , LAG(ts_duration) OVER (PARTITION BY tvid ORDER BY ts_start, ts_end) AS prev_duration
    , CASE WHEN next_start < ts_end AND DATE_TRUNC('HOUR', next_start) = DATE_TRUNC('HOUR', ts_end) THEN 1
           WHEN prev_end > ts_start AND DATE_TRUNC('HOUR', prev_end) = DATE_TRUNC('HOUR', ts_start) THEN 1
      END AS overlapping_session
    , CASE WHEN overlapping_session IS NULL THEN 1
           ELSE CASE WHEN next_start IS NOT NULL AND next_start < ts_end AND next_duration < ts_duration AND DATE_TRUNC('HOUR', next_start) = DATE_TRUNC('HOUR', ts_start) THEN 1
                     WHEN prev_end IS NOT NULL AND prev_end > ts_start AND prev_duration < ts_duration AND DATE_TRUNC('HOUR', prev_end) = DATE_TRUNC('HOUR', ts_end) THEN 1
        END END AS keep
    FROM (
      SELECT tvid
      , ts_start
      , ts_end
      , TIMESTAMPDIFF(SECOND, ts_start, ts_end) AS ts_duration
      , cid
      , chan_callsign
      , epid
      , ROW_NUMBER() OVER (PARTITION BY tvid, ts_start ORDER BY ts_start, ts_end DESC, epid) AS rn
      FROM prod.cooker.vizio_content_firehose vc
      WHERE ts_start >= '2025-05-11 11:00:00'
            AND ts_start < CURRENT_TIMESTAMP - INTERVAL 5 HOURS
      
    )
    WHERE rn = 1
  ) vc
  JOIN prod.detection.epg_station ism
    ON ism.station_call_sign = vc.chan_callsign
   AND ism.local_or_national = 'National'
  WHERE keep = 1
    AND DATE_TRUNC('HOUR', ts_start) != '2025-05-13 21:00:00'
  GROUP BY 1
)

In [0]:
%sql
(
  SELECT 'DP4' AS pipeline
  , COUNT(*)*1.0 AS sessions_count
  , SUM(TIMESTAMPDIFF(SECOND, ts_start, ts_end))/3600.0 AS total_duration
  , COUNT(Distinct vc.tvid) AS total_tvs
  FROM (
    SELECT *
    , LEAD(ts_start) OVER (PARTITION BY tvid ORDER BY ts_start, ts_end) AS next_start
    , LAG(ts_end) OVER (PARTITION BY tvid ORDER BY ts_start, ts_end) AS prev_end
    , LEAD(ts_duration) OVER (PARTITION BY tvid ORDER BY ts_start, ts_end) AS next_duration
    , LAG(ts_duration) OVER (PARTITION BY tvid ORDER BY ts_start, ts_end) AS prev_duration
    , CASE WHEN next_start < ts_end AND DATE_TRUNC('HOUR', next_start) = DATE_TRUNC('HOUR', ts_end) THEN 1
           WHEN prev_end > ts_start AND DATE_TRUNC('HOUR', prev_end) = DATE_TRUNC('HOUR', ts_start) THEN 1
      END AS overlapping_session
    , CASE WHEN overlapping_session IS NULL THEN 1
           ELSE CASE WHEN next_start IS NOT NULL AND next_start < ts_end AND next_duration < ts_duration AND DATE_TRUNC('HOUR', next_start) = DATE_TRUNC('HOUR', ts_start) THEN 1
                     WHEN prev_end IS NOT NULL AND prev_end > ts_start AND prev_duration < ts_duration AND DATE_TRUNC('HOUR', prev_end) = DATE_TRUNC('HOUR', ts_end) THEN 1
        END END AS keep
    FROM (
      SELECT tvid
      , ts_start
      , ts_end
      , TIMESTAMPDIFF(SECOND, ts_start, ts_end) AS ts_duration
      , cid
      , chan_callsign
      , epid
      , ROW_NUMBER() OVER (PARTITION BY tvid, ts_start ORDER BY ts_start, ts_end DESC, epid) AS rn
      FROM prod.staging.vizio_content_firehose
      WHERE ts_start >= '2025-05-11 11:00:00'
            AND ts_start < CURRENT_TIMESTAMP - INTERVAL 5 HOURS
      
    )
    WHERE rn = 1
  ) vc
  WHERE keep = 1
    AND DATE_TRUNC('HOUR', ts_start) != '2025-05-13 21:00:00'
  GROUP BY 1
)
UNION
(
  SELECT 'DP5' AS pipeline
  , COUNT(*)*1.0 AS sessions_count
  , SUM(TIMESTAMPDIFF(SECOND, ts_start, ts_end))/3600.0 AS total_duration
  , COUNT(Distinct vc.tvid) AS total_tvs
  FROM (
    SELECT *
    , LEAD(ts_start) OVER (PARTITION BY tvid ORDER BY ts_start, ts_end) AS next_start
    , LAG(ts_end) OVER (PARTITION BY tvid ORDER BY ts_start, ts_end) AS prev_end
    , LEAD(ts_duration) OVER (PARTITION BY tvid ORDER BY ts_start, ts_end) AS next_duration
    , LAG(ts_duration) OVER (PARTITION BY tvid ORDER BY ts_start, ts_end) AS prev_duration
    , CASE WHEN next_start < ts_end AND DATE_TRUNC('HOUR', next_start) = DATE_TRUNC('HOUR', ts_end) THEN 1
           WHEN prev_end > ts_start AND DATE_TRUNC('HOUR', prev_end) = DATE_TRUNC('HOUR', ts_start) THEN 1
      END AS overlapping_session
    , CASE WHEN overlapping_session IS NULL THEN 1
           ELSE CASE WHEN next_start IS NOT NULL AND next_start < ts_end AND next_duration < ts_duration AND DATE_TRUNC('HOUR', next_start) = DATE_TRUNC('HOUR', ts_start) THEN 1
                     WHEN prev_end IS NOT NULL AND prev_end > ts_start AND prev_duration < ts_duration AND DATE_TRUNC('HOUR', prev_end) = DATE_TRUNC('HOUR', ts_end) THEN 1
        END END AS keep
    FROM (
      SELECT tvid
      , ts_start
      , ts_end
      , TIMESTAMPDIFF(SECOND, ts_start, ts_end) AS ts_duration
      , cid
      , chan_callsign
      , epid
      , ROW_NUMBER() OVER (PARTITION BY tvid, ts_start ORDER BY ts_start, ts_end DESC, epid) AS rn
      FROM prod.cooker.vizio_content_firehose vc
      WHERE ts_start >= '2025-05-11 11:00:00'
            AND ts_start < CURRENT_TIMESTAMP - INTERVAL 5 HOURS
      
    )
    WHERE rn = 1
  ) vc
  WHERE keep = 1
    AND DATE_TRUNC('HOUR', ts_start) != '2025-05-13 21:00:00'
  GROUP BY 1
)

In [0]:
%sql
(
  SELECT 'DP4' AS pipeline
  , COUNT(*)*1.0 AS sessions_count
  , SUM(TIMESTAMPDIFF(SECOND, ts_start, ts_end))/3600.0 AS total_duration
  , COUNT(Distinct vc.tvid) AS total_tvs
  FROM (
    SELECT *
    , LEAD(ts_start) OVER (PARTITION BY tvid ORDER BY ts_start, ts_end) AS next_start
    , LAG(ts_end) OVER (PARTITION BY tvid ORDER BY ts_start, ts_end) AS prev_end
    , LEAD(ts_duration) OVER (PARTITION BY tvid ORDER BY ts_start, ts_end) AS next_duration
    , LAG(ts_duration) OVER (PARTITION BY tvid ORDER BY ts_start, ts_end) AS prev_duration
    , CASE WHEN next_start < ts_end AND DATE_TRUNC('HOUR', next_start) = DATE_TRUNC('HOUR', ts_end) THEN 1
           WHEN prev_end > ts_start AND DATE_TRUNC('HOUR', prev_end) = DATE_TRUNC('HOUR', ts_start) THEN 1
      END AS overlapping_session
    , CASE WHEN overlapping_session IS NULL THEN 1
           ELSE CASE WHEN next_start IS NOT NULL AND next_start < ts_end AND next_duration < ts_duration AND DATE_TRUNC('HOUR', next_start) = DATE_TRUNC('HOUR', ts_start) THEN 1
                     WHEN prev_end IS NOT NULL AND prev_end > ts_start AND prev_duration < ts_duration AND DATE_TRUNC('HOUR', prev_end) = DATE_TRUNC('HOUR', ts_end) THEN 1
        END END AS keep
    FROM (
      SELECT tvid
      , ts_start
      , ts_end
      , TIMESTAMPDIFF(SECOND, ts_start, ts_end) AS ts_duration
      , cid
      , chan_callsign
      , epid
      , ROW_NUMBER() OVER (PARTITION BY tvid, ts_start ORDER BY ts_start, ts_end DESC, epid) AS rn
      FROM prod.staging.vizio_content_firehose
      WHERE ts_start >= '2025-05-11 11:00:00'
            AND ts_start < CURRENT_TIMESTAMP - INTERVAL 5 HOURS
      
    )
    WHERE rn = 1
  ) vc
  WHERE keep = 1
    AND DATE_TRUNC('HOUR', ts_start) != '2025-05-13 21:00:00'
    AND LOWER(cid) != 'unknown'
  GROUP BY 1
)
UNION
(
  SELECT 'DP5' AS pipeline
  , COUNT(*)*1.0 AS sessions_count
  , SUM(TIMESTAMPDIFF(SECOND, ts_start, ts_end))/3600.0 AS total_duration
  , COUNT(Distinct vc.tvid) AS total_tvs
  FROM (
    SELECT *
    , LEAD(ts_start) OVER (PARTITION BY tvid ORDER BY ts_start, ts_end) AS next_start
    , LAG(ts_end) OVER (PARTITION BY tvid ORDER BY ts_start, ts_end) AS prev_end
    , LEAD(ts_duration) OVER (PARTITION BY tvid ORDER BY ts_start, ts_end) AS next_duration
    , LAG(ts_duration) OVER (PARTITION BY tvid ORDER BY ts_start, ts_end) AS prev_duration
    , CASE WHEN next_start < ts_end AND DATE_TRUNC('HOUR', next_start) = DATE_TRUNC('HOUR', ts_end) THEN 1
           WHEN prev_end > ts_start AND DATE_TRUNC('HOUR', prev_end) = DATE_TRUNC('HOUR', ts_start) THEN 1
      END AS overlapping_session
    , CASE WHEN overlapping_session IS NULL THEN 1
           ELSE CASE WHEN next_start IS NOT NULL AND next_start < ts_end AND next_duration < ts_duration AND DATE_TRUNC('HOUR', next_start) = DATE_TRUNC('HOUR', ts_start) THEN 1
                     WHEN prev_end IS NOT NULL AND prev_end > ts_start AND prev_duration < ts_duration AND DATE_TRUNC('HOUR', prev_end) = DATE_TRUNC('HOUR', ts_end) THEN 1
        END END AS keep
    FROM (
      SELECT tvid
      , ts_start
      , ts_end
      , TIMESTAMPDIFF(SECOND, ts_start, ts_end) AS ts_duration
      , cid
      , chan_callsign
      , epid
      , ROW_NUMBER() OVER (PARTITION BY tvid, ts_start ORDER BY ts_start, ts_end DESC, epid) AS rn
      FROM prod.cooker.vizio_content_firehose vc
      WHERE ts_start >= '2025-05-11 11:00:00'
            AND ts_start < CURRENT_TIMESTAMP - INTERVAL 5 HOURS
      
    )
    WHERE rn = 1
  ) vc
  WHERE keep = 1
    AND DATE_TRUNC('HOUR', ts_start) != '2025-05-13 21:00:00'
    AND LOWER(cid) != 'unknown'
  GROUP BY 1
)

In [0]:
%sql
SELECT token from dev.ayesha_sareen.dlm_update_apr25_100pct_xml_rollout_tv_test WHERE chipset = '5597'
GROUP BY 1
LIMIT 11